# Task 1.1 — Algorithm Explanation

**Paper:** *Analysis of SVM with Indefinite Kernels* (NIPS 2009)  
**Authors:** Yiming Ying, Colin Campbell, Mark Girolami

---

## Overview

This notebook provides a step-by-step explanation of the algorithm proposed in the paper for training SVMs when the kernel matrix is **indefinite** (i.e., not positive semi-definite).

Standard SVMs require PSD kernels to ensure the optimization problem is convex. This paper proposes a principled min-max framework that simultaneously optimizes the SVM classifier and learns a PSD proxy kernel close to the original indefinite kernel.

---

## Step 1: Problem Formulation for SVM with Indefinite Kernels

### Description

In standard SVM classification, we solve the dual optimization problem:

$$\max_{\alpha} \sum_{i=1}^{n} \alpha_i - \frac{1}{2} \sum_{i,j} \alpha_i \alpha_j y_i y_j K(x_i, x_j)$$

subject to $0 \leq \alpha_i \leq C$ and $\sum_i \alpha_i y_i = 0$.

This formulation assumes the kernel matrix $K$ is **positive semi-definite (PSD)**, which ensures the problem is convex. However, many practical similarity measures (e.g., sigmoidal kernel, power spectrum kernels, edit-distance based kernels) produce **indefinite** kernel matrices.

When $K$ is indefinite, the dual objective is no longer concave, meaning:
- The optimization problem may have multiple local optima
- Strong duality may not hold
- Standard QP solvers may not converge

### Equation Reference
Section 1, Introduction of the paper.

### Purpose
Motivates the need for a new formulation that can handle indefinite kernels in a principled manner.

---

## Step 2: Min-Max Formulation of Kernel Learning

### Description

The key insight of the paper is to reformulate the SVM problem as a **min-max optimization** over both the classifier parameters $\alpha$ and the kernel matrix $\tilde{K}$:

$$\min_{\alpha \in \mathcal{A}} \max_{\tilde{K} \in \mathcal{K}} \left[ \sum_{i=1}^{n} \alpha_i - \frac{1}{2} \alpha^T \tilde{K} \alpha \right]$$

where:
- $\mathcal{A} = \{\alpha : 0 \leq \alpha_i \leq C, \; \sum_i \alpha_i y_i = 0\}$ is the feasible set for dual variables
- $\mathcal{K} = \{\tilde{K} \succeq 0 : \|\tilde{K} - K\|_F \leq \rho\}$ is the set of PSD matrices within distance $\rho$ of the original kernel $K$

This formulation seeks the **worst-case PSD kernel** (maximization over $\tilde{K}$) while simultaneously finding the **best classifier** (minimization over $\alpha$).

### Equation Reference
Equation (3) in the paper; Section 2.

### Purpose
Provides a robust optimization framework that handles indefiniteness by jointly optimizing the kernel and classifier.

---

## Step 3: Objective Function $f(\alpha)$

### Description

After solving the inner maximization over $\tilde{K}$ (for a fixed $\alpha$), the problem reduces to minimizing:

$$f(\alpha) = \max_{\tilde{K} \in \mathcal{K}} \left[ \sum_{i=1}^{n} \alpha_i - \frac{1}{2} \alpha^T \tilde{K} \alpha \right]$$

The inner maximization has a closed-form solution: the optimal $\tilde{K}^*$ is the **projection of $K$ onto the PSD cone** (possibly with a scaling related to $\rho$).

This yields:

$$f(\alpha) = \mathbf{1}^T \alpha - \frac{1}{2} \alpha^T K_{+} \alpha$$

where $K_{+}$ denotes the positive part of $K$ obtained by clipping negative eigenvalues to zero.

### Equation Reference
Equations (4)-(6) in the paper; Theorem 1.

### Purpose
Reduces the min-max problem to a single minimization problem with a well-defined objective.

---

## Step 4: Projection of Kernel Matrix to PSD Cone

### Description

The projection of an indefinite kernel matrix $K$ onto the PSD cone is computed via **eigen-decomposition**:

1. Compute $K = Q \Lambda Q^T$ where $\Lambda = \text{diag}(\lambda_1, \ldots, \lambda_n)$
2. Clip negative eigenvalues: $\Lambda_{+} = \text{diag}(\max(\lambda_1, 0), \ldots, \max(\lambda_n, 0))$
3. Reconstruct: $K_{+} = Q \Lambda_{+} Q^T$

This is the **nearest PSD matrix** to $K$ in the Frobenius norm:

$$K_{+} = \arg\min_{\tilde{K} \succeq 0} \|K - \tilde{K}\|_F$$

### Equation Reference
Section 3; Proposition 1 in the paper.

### Purpose
Provides the mechanism to convert an indefinite kernel into a valid PSD kernel for use in the SVM dual.

---

## Step 5: Gradient Derivation

### Description

To optimize $f(\alpha)$ using gradient-based methods, we need the gradient:

$$\nabla f(\alpha) = \mathbf{1} - K_{+} \alpha$$

where $K_{+}$ is the PSD-projected kernel matrix.

The key theoretical result is that $f(\alpha)$ is **convex** (since $K_{+} \succeq 0$) and its gradient is **Lipschitz continuous** with constant $L = \|K_{+}\|$:

$$\|\nabla f(\alpha) - \nabla f(\beta)\| \leq L \|\alpha - \beta\|$$

This Lipschitz property is crucial for convergence guarantees of gradient methods.

### Equation Reference
Section 3; Lemma 2 and Theorem 2.

### Purpose
Enables the use of first-order optimization methods with provable convergence rates.

---

## Step 6: Simplified Projected Gradient Method

### Description

The paper proposes a **projected gradient method** for minimizing $f(\alpha)$:

**Algorithm:**
1. Initialize $\alpha^{(0)} = 0$
2. For $t = 0, 1, 2, \ldots$:
   - Compute gradient: $g^{(t)} = \nabla f(\alpha^{(t)}) = \mathbf{1} - K_{+} \alpha^{(t)}$
   - Update: $\hat{\alpha}^{(t+1)} = \alpha^{(t)} - \frac{1}{L} g^{(t)}$
   - Project onto feasible set: $\alpha^{(t+1)} = \Pi_{\mathcal{A}}(\hat{\alpha}^{(t+1)})$
3. Convergence: $f(\alpha^{(t)}) - f(\alpha^*) \leq \frac{L \|\alpha^{(0)} - \alpha^*\|^2}{2t}$

The projection $\Pi_{\mathcal{A}}$ ensures $\alpha$ remains in the feasible set (box constraints + equality constraint).

**Convergence rate:** $O(1/t)$ — sublinear but guaranteed.

### Equation Reference
Algorithm 1 in the paper; Section 4.

### Purpose
Provides a practical, implementable algorithm with convergence guarantees for the indefinite kernel SVM problem.

---

## Step 7: Nesterov Smooth Optimization

### Description

To achieve a faster convergence rate, the paper leverages **Nesterov's optimal gradient method**:

**Algorithm (Accelerated):**
1. Initialize $\alpha^{(0)} = \beta^{(0)} = 0$
2. For $t = 0, 1, 2, \ldots$:
   - Compute gradient at $\beta^{(t)}$: $g^{(t)} = \nabla f(\beta^{(t)})$
   - Gradient step: $\alpha^{(t+1)} = \Pi_{\mathcal{A}}(\beta^{(t)} - \frac{1}{L} g^{(t)})$
   - Momentum: $\beta^{(t+1)} = \alpha^{(t+1)} + \frac{t}{t+3}(\alpha^{(t+1)} - \alpha^{(t)})$

**Convergence rate:** $O(1/t^2)$ — optimal for first-order methods.

This accelerated method maintains the same per-iteration cost as the basic projected gradient but achieves quadratically faster convergence.

### Equation Reference
Algorithm 2 in the paper; Section 4; Theorem 3.

### Purpose
Achieves the optimal convergence rate for smooth convex optimization, making the algorithm practical for larger-scale problems.

---

## Summary

The paper introduces a principled **min-max framework** for training SVMs with indefinite kernels by projecting the kernel matrix onto the positive semi-definite cone via eigen-decomposition, and then solving the resulting convex optimization problem using projected gradient methods (with optional Nesterov acceleration), achieving provable convergence guarantees and enabling the use of non-PSD similarity measures in kernel-based classification.

| Step | Key Operation | Convergence |
|------|--------------|-------------|
| 1 | Problem formulation | — |
| 2 | Min-max reformulation | — |
| 3 | Objective $f(\alpha)$ | — |
| 4 | PSD projection via eigendecomp | $O(n^3)$ |
| 5 | Gradient computation | $O(n^2)$ per step |
| 6 | Projected gradient | $O(1/t)$ |
| 7 | Nesterov acceleration | $O(1/t^2)$ |